# Chapter 3: Axiomatic Derivations

This notebook follows Chapter 3 of [Boxes and Diamonds](https://bd.openlogicproject.org)
using the **Gamen.jl** package.

We cover:
- Substitution and tautology checking
- Axiom schemas (K, Dual, T, D, B, 4, 5) and instance matching
- Modal systems (K, KT, S4, S5, ...)
- Hilbert-style derivations and proof checking
- Dual formulas (Definition 3.26)
- Soundness (Theorem 3.31)

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using Gamen

## 3.1 Substitution

A *substitution* replaces propositional variables with arbitrary formulas.
This is the mechanism by which axiom *schemas* generate their *instances*:
a schema like `p → (q → p)` becomes `□r → (◇s → □r)` by substituting
`□r` for `p` and `◇s` for `q`.

In [ ]:
p = Atom(:p)
q = Atom(:q)
r = Atom(:r)

σ = Dict(p => □(r), q => ◇(q))
original = Implies(p, Implies(q, p))
substituted = substitute(original, σ)

println("Original:    ", original)
println("Substituted: ", substituted)

In [ ]:
# Substitution distributes through all connectives and modal operators
σ2 = Dict(p => □(q), q => ◇(r))

println("And:     ", substitute(And(p, q), σ2))
println("Box:     ", substitute(□(p), σ2))
println("Diamond: ", substitute(◇(Implies(p, q)), σ2))

## 3.2 Tautologies and Tautological Instances

A *propositional tautology* is a modal-free formula that is true under every
truth-value assignment. Gamen.jl checks this by exhaustive truth-table evaluation.

In [ ]:
# Classical tautologies
println("p ∨ ¬p:       ", is_tautology(Or(p, Not(p))))
println("¬¬p → p:      ", is_tautology(Implies(Not(Not(p)), p)))
println("p → (q → p):  ", is_tautology(Implies(p, Implies(q, p))))
println("⊤ (= ¬⊥):     ", is_tautology(Top()))

In [ ]:
# Non-tautologies
println("p:     ", is_tautology(p))
println("p → q: ", is_tautology(Implies(p, q)))
println("⊥:     ", is_tautology(Bottom()))

A *tautological instance* (Definition 3.3, item 1) is a formula whose
**propositional skeleton** is a tautology. The skeleton is obtained by
treating atoms and modal subformulas as propositional variables.

In [ ]:
# □p → □p is a tautological instance (skeleton: A → A)
println("□p → □p:             ", is_tautological_instance(Implies(□(p), □(p))))

# □p → (◇q → □p) is a tautological instance (skeleton: A → (B → A))
println("□p → (◇q → □p):      ", is_tautological_instance(Implies(□(p), Implies(◇(q), □(p)))))

# □p ∨ ¬□p (excluded middle with modal subformula)
println("□p ∨ ¬□p:            ", is_tautological_instance(Or(□(p), Not(□(p)))))

# □p → ◇q is NOT a tautological instance (skeleton: A → B)
println("□p → ◇q:             ", is_tautological_instance(Implies(□(p), ◇(q))))

## 3.3 Axiom Schemas

A modal logic is determined by its axiom schemas. Every normal modal logic
includes the **K** axiom and the **Dual** axiom. Additional schemas define
stronger systems.

| Schema | Formula | Meaning |
|:-------|:--------|:--------|
| **K** | □(A → B) → (□A → □B) | □ distributes over → |
| **Dual** | ◇A ↔ ¬□¬A | ◇ is the dual of □ |
| **T** | □A → A | Necessity implies truth |
| **D** | □A → ◇A | Necessity implies possibility |
| **B** | A → □◇A | Truth implies necessary possibility |
| **4** | □A → □□A | Necessity iterates |
| **5** | ◇A → □◇A | Possibility is necessary |

The function `is_instance` checks whether a formula matches a schema:

In [ ]:
# K axiom instance: □(p→q) → (□p→□q)
k_inst = Implies(□(Implies(p, q)), Implies(□(p), □(q)))
println("K instance:    ", k_inst, "  ", is_instance(k_inst, SchemaK()))

# K works with any subformulas
k_complex = Implies(
    □(Implies(And(p, q), ◇(r))),
    Implies(□(And(p, q)), □(◇(r))))
println("K (complex):   ", is_instance(k_complex, SchemaK()))

In [ ]:
# Check instances of all other schemas
println("Dual ◇(p∧q) ↔ ¬□¬(p∧q): ", is_instance(
    Iff(◇(And(p,q)), Not(□(Not(And(p,q))))), SchemaDual()))
println("T    □p → p:              ", is_instance(Implies(□(p), p), SchemaT()))
println("D    □p → ◇p:             ", is_instance(Implies(□(p), ◇(p)), SchemaD()))
println("B    p → □◇p:             ", is_instance(Implies(p, □(◇(p))), SchemaB()))
println("4    □p → □□p:            ", is_instance(Implies(□(p), □(□(p))), Schema4()))
println("5    ◇p → □◇p:            ", is_instance(Implies(◇(p), □(◇(p))), Schema5()))

In [ ]:
# Schema matching is strict — mismatched subformulas are rejected
println("□p → q  is T?  ", is_instance(Implies(□(p), q), SchemaT()))
println("□p → ◇q is D?  ", is_instance(Implies(□(p), ◇(q)), SchemaD()))
println("p → □◇q is B?  ", is_instance(Implies(p, □(◇(q))), SchemaB()))

## 3.4 Modal Systems

A **modal system** (Definition 3.9) is defined by its set of axiom schemas.
Gamen.jl provides eight standard systems:

In [ ]:
for sys in [SYSTEM_K, SYSTEM_KT, SYSTEM_KD, SYSTEM_KB,
            SYSTEM_K4, SYSTEM_K5, SYSTEM_S4, SYSTEM_S5]
    schemas = join(string.(sys.schemas), ", ")
    println("$(sys.name): $schemas")
end

## 3.5 Derivations and Proof Checking

A **derivation** (Definition 3.3) is a sequence of formulas where each step
is justified by one of four rules:

1. **Tautological instance** — the propositional skeleton is a tautology
2. **Axiom instance** — the formula matches an axiom schema of the system
3. **Modus ponens** — from A and A → B, derive B
4. **Necessitation** — from A, derive □A

### Proof of Proposition 3.12: □A → □(B → A)

This is a four-step proof in system K:

In [ ]:
proof_312 = Derivation([
    # Step 1: A → (B → A) is a tautological instance
    ProofStep(Implies(p, Implies(q, p)), Tautology()),
    # Step 2: □(A → (B → A)) by necessitation from step 1
    ProofStep(□(Implies(p, Implies(q, p))), Necessitation(1)),
    # Step 3: K axiom instance
    ProofStep(
        Implies(□(Implies(p, Implies(q, p))),
                Implies(□(p), □(Implies(q, p)))),
        AxiomInst(SchemaK())),
    # Step 4: Modus ponens from steps 2 and 3
    ProofStep(
        Implies(□(p), □(Implies(q, p))),
        ModusPonens(2, 3)),
])

println(proof_312)
println()
println("Valid in K: ", is_valid_derivation(SYSTEM_K, proof_312))
println("Conclusion: ", conclusion(proof_312))

### Proof of Proposition 3.13: □(A ∧ B) → (□A ∧ □B)

This longer proof first derives □(A∧B) → □A and □(A∧B) → □B separately,
then combines them using a propositional tautology:

In [ ]:
ab = And(p, q)

proof_313 = Derivation([
    # Part 1: □(A∧B) → □A
    ProofStep(Implies(ab, p), Tautology()),                              # 1
    ProofStep(□(Implies(ab, p)), Necessitation(1)),                      # 2
    ProofStep(Implies(□(Implies(ab, p)),
                      Implies(□(ab), □(p))), AxiomInst(SchemaK())),      # 3
    ProofStep(Implies(□(ab), □(p)), ModusPonens(2, 3)),                  # 4

    # Part 2: □(A∧B) → □B
    ProofStep(Implies(ab, q), Tautology()),                              # 5
    ProofStep(□(Implies(ab, q)), Necessitation(5)),                      # 6
    ProofStep(Implies(□(Implies(ab, q)),
                      Implies(□(ab), □(q))), AxiomInst(SchemaK())),      # 7
    ProofStep(Implies(□(ab), □(q)), ModusPonens(6, 7)),                  # 8

    # Combine via (p→q) → ((p→r) → (p→(q∧r)))
    ProofStep(
        Implies(Implies(□(ab), □(p)),
                Implies(Implies(□(ab), □(q)),
                        Implies(□(ab), And(□(p), □(q))))),
        Tautology()),                                                     # 9
    ProofStep(
        Implies(Implies(□(ab), □(q)),
                Implies(□(ab), And(□(p), □(q)))),
        ModusPonens(4, 9)),                                               # 10
    ProofStep(
        Implies(□(ab), And(□(p), □(q))),
        ModusPonens(8, 10)),                                              # 11
])

println("Valid in K: ", is_valid_derivation(SYSTEM_K, proof_313))
println("Conclusion: ", conclusion(proof_313))

### Invalid Derivations

The proof checker catches errors — wrong references, axioms not in the
system, and mismatched formulas:

In [ ]:
# Schema T is not available in system K, but it IS valid in KT
bad_proof = Derivation([
    ProofStep(Implies(□(p), p), AxiomInst(SchemaT())),
])

println("Valid in K:  ", is_valid_derivation(SYSTEM_K, bad_proof))
println("Valid in KT: ", is_valid_derivation(SYSTEM_KT, bad_proof))

In [ ]:
# Wrong modus ponens: can't derive p from (p→p) alone
bad_mp = Derivation([
    ProofStep(Implies(p, p), Tautology()),
    ProofStep(p, ModusPonens(1, 1)),
])
println("Valid: ", is_valid_derivation(SYSTEM_K, bad_mp))

## 3.6 Dual Formulas (Definition 3.26)

The *dual* of a formula is obtained by swapping:
- ⊥ ↔ ⊤
- ∧ ↔ ∨
- □ ↔ ◇

Atoms are negated, and negation distributes through the dual.
The key property is that ⊢ A ↔ ¬(dual(A)).

In [ ]:
println("dual(⊥)     = ", dual(Bottom()))
println("dual(p)     = ", dual(p))
println("dual(¬p)    = ", dual(Not(p)))
println("dual(p ∧ q) = ", dual(And(p, q)))
println("dual(p ∨ q) = ", dual(Or(p, q)))
println("dual(□p)    = ", dual(□(p)))
println("dual(◇p)    = ", dual(◇(p)))

In [ ]:
# Nested example: dual of □(p ∧ q) = ◇(¬p ∨ ¬q)
nested = □(And(p, q))
println("Original: ", nested)
println("Dual:     ", dual(nested))

## 3.7 Soundness (Theorem 3.31)

If a formula is provable in a modal system, it is valid on the
corresponding class of frames. We can verify this semantically:
K-provable formulas should be valid on *all* frames, and
system-specific theorems should be valid on the appropriate frame class.

In [ ]:
# □p → □(q → p) is K-provable — valid on any frame
thm = Implies(□(p), □(Implies(q, p)))

f1 = KripkeFrame([:w1, :w2], [:w1 => :w2])
f2 = KripkeFrame([:w1], [:w1 => :w1])
f3 = KripkeFrame([:w1, :w2, :w3], [:w1 => :w2, :w2 => :w3])

println("□p → □(q → p) valid on:")
println("  linear 2-frame:   ", is_valid_on_frame(f1, thm))
println("  reflexive 1-frame: ", is_valid_on_frame(f2, thm))
println("  linear 3-frame:   ", is_valid_on_frame(f3, thm))

In [ ]:
# Schema T (□p → p) — valid on reflexive frames only
schema_t = Implies(□(p), p)

reflexive = KripkeFrame([:w1, :w2], [:w1 => :w1, :w2 => :w2, :w1 => :w2])
non_reflexive = KripkeFrame([:w1, :w2], [:w1 => :w2])

println("□p → p valid on:")
println("  reflexive frame:     ", is_valid_on_frame(reflexive, schema_t))
println("  non-reflexive frame: ", is_valid_on_frame(non_reflexive, schema_t))

## Exploring on Your Own

Try these exercises:

- Construct a proof of □(A ∧ B) ← (□A ∧ □B) — the converse of Proposition 3.13
- Verify that ◇(A ∨ B) ↔ (◇A ∨ ◇B) is valid on all frames (Proposition 3.15)
- Build a derivation using Schema T in system KT
- Compute the dual of □(p → q) and verify that the original and ¬dual are equivalent on a model

In [ ]:
# Scratch space for exercises